In [37]:
from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from pydantic import BaseModel, Field
import pandas as pd
from config import API_KEY

pd.set_option('display.max_colwidth', None)

In [38]:
item = pd.read_csv('./data/preprocessed_item.csv')
youtube = pd.read_csv('./data/youtube_data.csv', lineterminator='\n')

In [39]:
# 토큰 리미트 초과 방지
# youtube['video_description'] = youtube['video_description'].fillna('')

# def split_str(column, max_length=6000):
#     return column.apply(lambda x: [x[i:i + max_length] for i in range(0, len(x), max_length)])

# youtube['desc_split'] = split_str(youtube['video_description'], 3000)
# youtube = youtube.explode('desc_split', ignore_index=True)

In [40]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key = API_KEY)

# Pydantic
class Extract(BaseModel):
    cosmetic_list: list = Field(description="The list of the cosmetics names")

structured_llm = llm.with_structured_output(Extract)

In [41]:
examples = [
    HumanMessage("#shorts #올리브영아이라이너 #지속력좋은아이라이너 #클리오샤프쏘심플워터프루프펜슬라이너 #코스노리슈퍼프루프피팅젤아이라이너 #머지더퍼스트펜아이라이너 #클리오워터프루프펜라이너킬브라운 #웨이크메이크철벽펜아이라이너", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "cosmetic_list": ["클리오샤프쏘심플워터프루프펜슬라이너", "코스노리슈퍼프루프피팅젤아이라이너", "머지더퍼스트펜아이라이너", "클리오워터프루프펜라이너킬브라운", "웨이크메이크철벽펜아이라이너"]},
                "id": "1",
            }
        ],
    ),
    ToolMessage("", tool_call_id="1"),
    HumanMessage("💛 한율 달빛유자 패드 X 올리브영 프로모션 (~11/29) 💛🔎 5분 잡티톤업 에센스 패드: https://bit.ly/3ZkzK8E 정가 29,000원 → 23,200원 (20% OFF) #한율 #달빛유자패드 #5분에센스패드 #비타민패드", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "cosmetic_list": ["한율 달빛유자 패드", "5분 잡티톤업 에센스 패드"]},
                "id": "2",
            }
        ],
    ),
    ToolMessage("", tool_call_id="2"),
    HumanMessage("👉🏻SKINFOOD : 스킨푸드 00:52 연어 다크서클 컨실러 3colors / 8,000원 직접구매 🔗https://www.oliveyoung.co.kr/store/go...  👉🏻Dr.Jart+ : 닥터자르트 02:28 시카페어 인텐시브 수딩리페어 크림 50ml / 50,000원 직접구매 🔗https://www.oliveyoung.co.kr/store/go... " + \
                "👉🏻MAMONDE : 마몽드 04:35 플로라 글로우 로즈 리퀴드 마스크 80ml / 30,000원 직접구매 (직접구매해서 쭉 사용하다가 광고도 받았던 제품!) 🔗https://www.oliveyoung.co.kr/store/go...", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "cosmetic_list": ["연어 다크서클 컨실러", "시카페어 인텐시브 수딩리페어 크림", "플로라 글로우 로즈 리퀴드 마스크"]},
                "id": "3",
            }
        ],
    ),
    ToolMessage("", tool_call_id="3"),
]
# 시스템 프롬프트
system = """주어진 텍스트에서 화장품 이름을 모두 찾아주세요.
    반환 형식은 JSON으로, "cosmetic_list" 키를 사용해 반환합니다."""

# ChatPromptTemplate 생성
prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("placeholder", "{examples}"), ("human", "{input}")]
)

# 모델과 프롬프트 연결
few_shot_structured_llm = prompt | structured_llm

def invoke(input):
    result = few_shot_structured_llm.invoke({"input": input, "examples": examples})
    return result.cosmetic_list

In [42]:
youtube['product'] = youtube['video_description'].apply(invoke)

In [ ]:
#youtube.to_csv('youtube_extract.csv', index=False)

In [ ]:
youtube2 = youtube.explode('product')

In [ ]:
#중복 제거
youtube2 = youtube2.drop_duplicates()

1243
0      False
0      False
0      False
0      False
0      False
       ...  
209    False
209    False
209    False
209    False
209    False
Length: 1193, dtype: bool
1193


In [ ]:
# 유효하지않은 값 삭제
# stopword 지정
words = []

for idx, row in item.iterrows():
    words.extend(row['item'].split())

stopword = set(words)


In [ ]:
from rapidfuzz import process, fuzz

def matching(word):
    if word in stopword:
        return None
    ans = process.extractOne(word, item['item'], score_cutoff=90)
    if ans:
        return ans[0]
    else:
        ans

In [ ]:
youtube2['match'] = youtube2['product'].apply(matching)

In [ ]:
youtube2 = youtube2.reset_index()
youtube2.rename(columns={'index': 'video_index'}, inplace=True)

In [16]:
youtube2.to_csv('youtube_match.csv', index=False)